> ⚠️ **Sandbox notice** — This notebook runs against the FiGuard shared demo environment (`sb_live_demo` key, `https://figuard-sandbox-g1ha.onrender.com`). It is **not for production use**. Budgets, events, and tokens may be wiped at any time without notice to keep the sandbox available for everyone. No uptime SLA. [Self-host for production →](https://github.com/figuard/figuard-core#self-hosting)

> 💡 **Quick start:** Select **Runtime → Run all** (Ctrl+F9) to run everything at once, then read the output below.

In [ ]:
# @title Step 1 — Install and configure FiGuard
import sys
!pip install --upgrade "figuard[openai-agents]>=0.3.0" openai-agents -q
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} + OpenAI Agents ready")
import uuid
USER_ID = f"demo_{uuid.uuid4().hex[:8]}"
print(f"Your session ID: {USER_ID}  (identifies your budgets in the dashboard)")

In [ ]:
from agents import Agent, Runner, function_tool
from figuard.integrations.openai_agents import guarded_function_tool
from figuard import FiGuardClient

client = FiGuardClient()

budget = client.create_budget(
    user_id=USER_ID,
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="travel booking session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit: ${budget.total_limit}")

session_token = budget.session_token

In [ ]:
# guarded_function_tool wraps the raw function BEFORE @function_tool converts
# it to a tool schema — decorator order matters.
#
# Step 1: define the guarded callable (holds a reference we can call directly)
@guarded_function_tool(
    client=client,
    session_token=session_token,
    category="flight",
    amount_key="amount",
    agent_id="travel_agent",
)
def _book_flight(description: str, amount: float) -> str:
    """Book a flight for the given description and amount."""
    return f"Flight booked: {description} for ${amount:.2f}"

# Step 2: wrap in @function_tool for the OpenAI Agents SDK
book_flight = function_tool(_book_flight)

agent = Agent(
    name="travel_agent",
    instructions="Help the user book flights within their budget.",
    tools=[book_flight],
)

# Full agent run (requires OPENAI_API_KEY):
# result = Runner.run_sync(agent, "Book me a flight from SFO to JFK for $270")
# print(result.final_output)

# ── Demo: call the guarded function directly — no OpenAI key needed ──────────
print("Simulating two tool calls from the agent:\n")

# Call 1: within budget → should authorize and confirm
r1 = _book_flight(description="SFO → JFK roundtrip", amount=270.0)
print(f"Tool call 1: {r1}")

# Call 2: over remaining budget → should be denied
r2 = _book_flight(description="SFO → LHR business class", amount=450.0)
print(f"Tool call 2: {r2}")

# Show final budget state
b = client.get_budget(budget.id)
print(f"\nBudget: ${b.quantity_spent:.2f} spent, ${b.available_quantity:.2f} remaining of ${b.total_limit:.2f}")
print(f"\n→ See the full spend tree: https://figuard-sandbox-g1ha.onrender.com/ui/budgets/{budget.id}")